# BakedAvatar — Kaggle Notebook

**BakedAvatar: Baking Neural Fields for Real-Time Head Avatar Synthesis**  
Paper: https://arxiv.org/abs/2311.05521 | Repo: https://github.com/buaavrcg/BakedAvatar

## Before you start
1. **GPU**: Set Accelerator to **T4 GPU** (or 2×T4 for multi-GPU training) in Settings
2. **Internet**: Enable internet access in Settings → Internet
3. **FLAME model**: The FLAME 2020 model requires registration at https://flame.is.tue.mpg.de/download.php  
   After downloading, create a Kaggle dataset containing `generic_model.pkl` and attach it here.
   Alternatively, [FLAME 2023 Open](https://flame.is.tue.mpg.de/download.php) (free) has a compatible structure — rename `flame2023_Open.pkl` to `generic_model.pkl`.

## Stages
- **Stage 1**: Train implicit neural fields (~30k iterations)
- **Stage 2**: Bake meshes and textures from trained fields
- **Stage 3**: Fine-tune textures at high resolution
- **Inference**: Evaluate / render with trained model

## 1. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU count: {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')

## 2. Install Dependencies

In [ ]:
%%bash
pip install -q accelerate configargparse chumpy opencv-python pymeshlab trimesh \
               scikit-image xatlas matplotlib tensorboard tqdm torchmetrics \
               face-alignment gdown Cython 2>&1 | tail -5

In [ ]:
%%bash
# Install nvdiffrast (requires CUDA)
git clone --depth 1 https://github.com/NVlabs/nvdiffrast /tmp/nvdiffrast 2>/dev/null || true
pip install -q --no-build-isolation /tmp/nvdiffrast 2>&1 | tail -5

In [ ]:
%%bash
# Install pytorch3d — try pre-built wheel first, fall back to source
TORCH=$(python -c "import torch; v=torch.__version__; print(v.split('+')[0])")
CUDA=$(python -c "import torch; print('cu' + torch.version.cuda.replace('.',''))" 2>/dev/null || echo 'cpu')
PY=$(python -c "import sys; print(f'cp{sys.version_info.major}{sys.version_info.minor}')")

echo "Trying pre-built wheel for torch=${TORCH}, ${CUDA}, ${PY}..."
pip install -q "pytorch3d" 2>/dev/null && echo 'Installed from PyPI' && exit 0

echo 'Pre-built not available, building from source (this takes ~5 min)...'
pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/pytorch3d.git' 2>&1 | tail -10

In [ ]:
import torch, pytorch3d, nvdiffrast, trimesh, accelerate
print('torch:', torch.__version__)
print('pytorch3d:', pytorch3d.__version__)
print('nvdiffrast: OK')
print('All dependencies installed successfully!')

## 3. Clone Repository & Install libmise

In [ ]:
%%bash
if [ ! -d '/kaggle/working/BakedAvatar' ]; then
    git clone https://github.com/buaavrcg/BakedAvatar /kaggle/working/BakedAvatar
else
    echo 'Repo already cloned'
fi

In [ ]:
%%bash
pip install -q --no-build-isolation /kaggle/working/BakedAvatar/code/utils/libmise/ 2>&1 | tail -5

In [ ]:
# Apply compatibility patches
import re

# Patch 1: metrics.py — auto-detect device for face_alignment
metrics_path = '/kaggle/working/BakedAvatar/code/model/metrics.py'
with open(metrics_path) as f:
    src = f.read()

old = '''        try:
            self.fa = face_alignment.FaceAlignment(face_alignment.LandmarksType.TWO_D, flip_input=False)
        except:
            self.fa = face_alignment.FaceAlignment(face_alignment.LandmarksType._2D, flip_input=False)'''
new = '''        device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
        try:
            self.fa = face_alignment.FaceAlignment(face_alignment.LandmarksType.TWO_D, flip_input=False, device=device)
        except:
            self.fa = face_alignment.FaceAlignment(face_alignment.LandmarksType._2D, flip_input=False, device=device)'''

if old in src:
    with open(metrics_path, 'w') as f:
        f.write(src.replace(old, new))
    print('Patched metrics.py')
else:
    print('metrics.py already patched or different version')

# Patch 2: dataset/real.py — fix float64 keypoints
dataset_path = '/kaggle/working/BakedAvatar/code/dataset/real.py'
with open(dataset_path) as f:
    src = f.read()

old2 = 'halfsize_bbox = np.array([img_res[0], img_res[1], img_res[0], img_res[1]]) / 2'
new2 = 'halfsize_bbox = np.array([img_res[0], img_res[1], img_res[0], img_res[1]], dtype=np.float32) / 2'

if old2 in src:
    with open(dataset_path, 'w') as f:
        f.write(src.replace(old2, new2))
    print('Patched dataset/real.py')
else:
    print('dataset/real.py already patched or different version')

## 4. FLAME Model Setup

**Option A (recommended):** Upload FLAME 2020 or FLAME 2023 Open as a Kaggle dataset:
1. Register at https://flame.is.tue.mpg.de/download.php
2. Download FLAME 2020 and extract `generic_model.pkl`  
   (or use FLAME 2023 Open and rename `flame2023_Open.pkl` → `generic_model.pkl`)
3. Create a Kaggle dataset with this file and attach it to this notebook
4. Update `FLAME_MODEL_SOURCE` below to point to the attached dataset path

**Option B:** Upload the file directly to `/kaggle/working/`

In [ ]:
import os, shutil

# Update this path to your FLAME model location
# Option A: from attached Kaggle dataset
FLAME_MODEL_SOURCE = '/kaggle/input/flame-model/generic_model.pkl'
# Option B: uploaded directly
# FLAME_MODEL_SOURCE = '/kaggle/working/generic_model.pkl'

FLAME_DEST = '/kaggle/working/BakedAvatar/code/flame/FLAME2020/generic_model.pkl'
os.makedirs(os.path.dirname(FLAME_DEST), exist_ok=True)

if os.path.exists(FLAME_MODEL_SOURCE):
    shutil.copy(FLAME_MODEL_SOURCE, FLAME_DEST)
    print(f'FLAME model copied to {FLAME_DEST}')
else:
    print(f'ERROR: FLAME model not found at {FLAME_MODEL_SOURCE}')
    print('Please attach your FLAME Kaggle dataset or update FLAME_MODEL_SOURCE')

## 5. Download Training Data & Pretrained Checkpoint

In [ ]:
%%bash
cd /kaggle/working/BakedAvatar
mkdir -p data/datasets data/experiments

# Download subject1 dataset (~3.5 GB)
if [ ! -d 'data/datasets/yufeng' ]; then
    echo 'Downloading subject1 dataset...'
    wget -q --show-progress \
        https://dataset.ait.ethz.ch/downloads/IMavatar_data/data/subject1.zip \
        -O data/datasets/subject1.zip
    unzip -q data/datasets/subject1.zip -d data/datasets/
    # subject1.zip extracts as subject1/subject1/MVI_*
    ln -sf /kaggle/working/BakedAvatar/data/datasets/subject1/subject1 \
           /kaggle/working/BakedAvatar/data/datasets/yufeng
    echo 'Dataset ready at data/datasets/yufeng'
else
    echo 'Dataset already present'
fi

In [ ]:
%%bash
cd /kaggle/working/BakedAvatar

# Download pretrained checkpoint (~3.5 GB 7zip archive)
if [ ! -d 'data/experiments/subject1/checkpoints' ]; then
    echo 'Downloading pretrained checkpoint...'
    gdown 137TTr8GENZmPZ-Me1SatymMCiVDKqDPR -O data/pretrained.7z

    # Install 7zip if needed
    apt-get install -qq p7zip-full
    7z x data/pretrained.7z -odata/ -y > /dev/null

    # The archive extracts as yufeng/034_8layer_.../
    mv data/yufeng/034_8layer_Rfreq12_Mfreq5_flamegtdist_eyeweight2 data/experiments/subject1
    echo "Checkpoint ready at data/experiments/subject1/"
    ls data/experiments/subject1/checkpoints/
else
    echo 'Checkpoint already present'
fi

## 6. Configure Accelerate

Choose **single GPU** or **multi-GPU** (2×T4) below.

In [ ]:
import torch, os

num_gpus = torch.cuda.device_count()
print(f'Detected {num_gpus} GPU(s)')

# Choose multi_gpu or single
use_multi_gpu = num_gpus > 1

os.makedirs(os.path.expanduser('~/.cache/huggingface/accelerate'), exist_ok=True)
config_path = os.path.expanduser('~/.cache/huggingface/accelerate/default_config.yaml')

if use_multi_gpu:
    config = f'''compute_environment: LOCAL_MACHINE
distributed_type: MULTI_GPU
mixed_precision: 'fp16'
num_machines: 1
num_processes: {num_gpus}
use_cpu: false
'''
    print(f'Configured for multi-GPU ({num_gpus} GPUs, fp16 mixed precision)')
else:
    config = '''compute_environment: LOCAL_MACHINE
distributed_type: NO
mixed_precision: 'no'
num_machines: 1
num_processes: 1
use_cpu: false
'''
    print('Configured for single GPU')

with open(config_path, 'w') as f:
    f.write(config)
print(f'Config written to {config_path}')

## 7. Run Inference (Stage-1 Implicit Fields)

Evaluates the pretrained Stage-1 model. Outputs are saved to `data/experiments/subject1/test/`.

In [ ]:
%%bash
cd /kaggle/working/BakedAvatar/code

accelerate launch scripts/runner.py \
    -c config/subject1.yaml \
    -t test \
    --img_res 512 512

## 8. Train from Scratch (Stage-1)

Full training takes ~30k iterations (~8-12 hours on T4). Uncomment to run.

In [ ]:
# %%bash
# cd /kaggle/working/BakedAvatar/code
#
# accelerate launch scripts/runner.py \
#     -c config/subject1.yaml \
#     -t train

## 9. Bake Meshes & Textures (Stage-2)

Requires Stage-1 trained model. Requires nvdiffrast (CUDA). Uncomment to run.

In [ ]:
# %%bash
# cd /kaggle/working/BakedAvatar/code
#
# # Extract meshes
# accelerate launch scripts/runner.py -c config/subject1.yaml -t mesh_export
#
# # Bake textures
# MESH_PKL=../data/experiments/subject1/mesh_export/iter_30000/marching_cube/res_init16_up5/mesh_data.pkl
# accelerate launch scripts/runner.py -c config/subject1.yaml -t texture_export \
#     --mesh_data_path $MESH_PKL

## 10. Fine-Tune Textures (Stage-3)

Requires Stage-2 baked meshes. Uncomment to run.

In [ ]:
# %%bash
# cd /kaggle/working/BakedAvatar/code
#
# MESH_PKL=../data/experiments/subject1/mesh_export/iter_30000/marching_cube/res_init16_up5/mesh_data.pkl
# accelerate launch scripts/runner.py -c config/subject1.yaml -t fine_tuning \
#     --img_res 512 512 --batch_size 1 \
#     --mesh_data_path $MESH_PKL

## 11. Evaluate Baked / Fine-Tuned Meshes (Stage-2/3)

Requires Stage-2 or Stage-3 output. Uncomment to run.

In [ ]:
# %%bash
# cd /kaggle/working/BakedAvatar/code
#
# # Evaluate fine-tuned meshes (Stage-3)
# FINETUNE_PKL=../data/experiments/subject1/finetune_mesh_data/iter_30000/mesh_data.pkl
# accelerate launch scripts/runner.py -c config/subject1.yaml -t test \
#     --img_res 512 512 --use_finetune_model \
#     --mesh_data_path $FINETUNE_PKL

## 12. View Results

In [ ]:
import os, glob
from IPython.display import Image, display
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Find test output images
test_dir = '/kaggle/working/BakedAvatar/data/experiments/subject1'
image_dirs = sorted(glob.glob(os.path.join(test_dir, 'test*', '**'), recursive=True))
png_files = sorted(glob.glob(os.path.join(test_dir, '**', '*.png'), recursive=True))[:6]

if png_files:
    fig, axes = plt.subplots(1, min(len(png_files), 6), figsize=(18, 4))
    if len(png_files) == 1:
        axes = [axes]
    for ax, path in zip(axes, png_files):
        img = mpimg.imread(path)
        ax.imshow(img)
        ax.set_title(os.path.basename(path), fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No output images found yet. Run inference first.')
    print('Looking in:', test_dir)